## Setup

In [1]:
from string import ascii_uppercase

import matplotlib.pyplot as plt
import numpy as np
from tigramite import data_processing as pp
from tigramite import plotting as tp
from tigramite.independence_tests.gsquared import Gsquared
from tigramite.lpcmci import LPCMCI

from csi_vae_gumbel.settings import Settings

settings = Settings()

ACTIVITIES_IDS = [f"S1a_{x}" for x in ascii_uppercase[: settings.n_activities]]
ACTIVITIES_LABELS = settings.activities
TIME_STEP = 15
GPU_ID = 0

## Dataset

We use the complete dataset for causal inference, withouy any train/test split.

In [6]:
latents = np.load(f"../{settings.study_path}/latents/latents.npy")
labels = np.load(f"../{settings.study_path}/latents/labels.npy")

labelled_causal_data = {}

# 2. Split and THEN shift per activity
for label in np.unique(labels):
    # Extract latents for this specific movement sequence
    activity_latents = latents[labels == label]

    # Create the shifted views locally for this activity
    # This ensures t-1 and t-2 belong to the SAME activity
    data_t = activity_latents[2:]  # current
    data_t_1 = activity_latents[1:-1]  # lag 1
    data_t_2 = activity_latents[:-2]  # lag 2

    # Horizontal stack: (T-2, latent_dim * 3)
    # Row format: [Z_0...Z_dim (t), Z_0...Z_dim (t-1), Z_0...Z_dim (t-2)]
    activity_causal_data = np.hstack([data_t, data_t_1, data_t_2])

    labelled_causal_data[int(label)] = activity_causal_data

## Causal Analysis

In [ ]:
# 1. Setup the analysis parameters
# Use Gsquared for categorical (discrete) data
g_test = Gsquared()
tau_max = 2

# Correct names: We have 'latent_dim' variables, each taking 'n_categories' values
var_names = [f"Z{i}" for i in range(latents.shape[1])]

# 2. Iterate through each activity
for label_id, data in labelled_causal_data.items():
    # 'data' should be shape (T, latent_dim) containing the argmax indices
    activity_name = ACTIVITIES_LABELS[label_id]

    if len(data) <= tau_max:
        print(f"Skipping {activity_name}: Not enough samples.")
        continue

    # Initialize Tigramite DataFrame
    # Note: 'datatypes' should be 'discrete' for Gsquared
    dataframe = pp.DataFrame(data.astype(np.int32), var_names=var_names, data_type=np.ones_like(data, dtype=int))

    # 3. Run LPCMCI
    # LPCMCI is robust to latent common causes (unobserved drivers in CSI)
    lpcmci = LPCMCI(dataframe=dataframe, cond_ind_test=g_test, verbosity=0)

    # pc_alpha is the significance level for the conditional independence tests
    results = lpcmci.run_lpcmci(tau_max=tau_max, pc_alpha=0.05)

    # 4. Plotting
    # Time series graph is best for t, t-1, t-2 visualization
    fig, ax = tp.plot_time_series_graph(
        val_matrix=results["val_matrix"],
        graph=results["graph"],
        var_names=var_names,
        link_colorbar_label="MCI Strength",
    )

    plt.title(f"Temporal Latent Causal Graph: {activity_name}")
    plt.show()